In [1]:
import os

from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model='deepseek-ai/DeepSeek-V3.2',
    base_url=os.environ.get('OPENAI_BASE_URL'),
    api_key=os.environ.get('OPENAI_API_KEY')
)

from langchain_community.utilities import SQLDatabase

hostname = 'localhost'
port = '3306'
database = 'pai_coding'
username = 'root'
password = '123456'
mysql_uri = f'mysql+pymysql://{username}:{password}@{hostname}:{port}/{database}?charset=utf8mb4'
db = SQLDatabase.from_uri(mysql_uri);

In [2]:
from langchain_core.messages import SystemMessage

system_prompt = """
您是一个被设计用来与SQL数据库交互的代理。
给定一个输入问题，创建一个语法正确的SQL语句并执行，然后查看查询结果并返回答案。
除非用户指定了他们想要获得的示例的具体数量，否则始终将SQL查询限制为最多10个结果。
你可以按相关列对结果进行排序，以返回MySQL数据库中最匹配的数据。
您可以使用与数据库交互的工具。在执行查询之前，你必须仔细检查。如果在执行查询时出现错误，请重写查询SQL并重试。
不要对数据库做任何DML语句(插入，更新，删除，删除等)。

首先，你应该查看数据库中的表，看看可以查询什么。
不要跳过这一步。
然后查询最相关的表的模式。
"""
system_message = SystemMessage(content=system_prompt)

In [4]:
from langchain.agents import create_agent
from langchain_community.agent_toolkits import SQLDatabaseToolkit

toolkit = SQLDatabaseToolkit(db=db, llm=model)
tools = toolkit.get_tools()
agent_executor = create_agent(model, tools)

In [6]:
from langchain_core.messages import HumanMessage
resp = agent_executor.invoke({
    "messages": [HumanMessage(content="请问一共有多少篇文章")],
})
resp

{'messages': [HumanMessage(content='请问一共有多少篇文章', additional_kwargs={}, response_metadata={}, id='6a19728d-4e89-4a32-a4f7-21a80a6b4ffc'),
  AIMessage(content='我来帮您查询数据库中的文章数量。首先让我查看一下数据库中有哪些表。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 62, 'prompt_tokens': 694, 'total_tokens': 756, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 694}, 'model_provider': 'openai', 'model_name': 'deepseek-ai/DeepSeek-V3.2', 'system_fingerprint': '', 'id': '019d1b3b1e7d1e716b10f59857e4bf00', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d1b3b-1d09-78f3-bafe-2a3567a5ab5a-0', tool_calls=[{'name': 'sql_db_list_tables', 'args': {'tool_input': ''}, 'id': '019d1b3b310fce3a8e1e276e3fb83123', 'type': 'tool_call'}], invalid

In [7]:
resp = agent_executor.invoke({
    "messages": [HumanMessage(content="哪篇文章最有意思")],
})
resp

{'messages': [HumanMessage(content='哪篇文章最有意思', additional_kwargs={}, response_metadata={}, id='5fa804bf-1146-4e19-8a17-0e355cd9bf1b'),
  AIMessage(content='这个问题需要使用数据库查询工具来获取相关数据。首先让我查看一下数据库中有哪些表，然后了解表结构，最后再查询哪篇文章最有意思。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 76, 'prompt_tokens': 694, 'total_tokens': 770, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 694}, 'model_provider': 'openai', 'model_name': 'deepseek-ai/DeepSeek-V3.2', 'system_fingerprint': '', 'id': '019d1b3caa37d80401c1c86dbdb207b3', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d1b3c-a8b3-7fa3-8761-dbb91a56efa3-0', tool_calls=[{'name': 'sql_db_list_tables', 'args': {'tool_input': ''}, 'id': '019d1b3cc42b3c1a78f82882a7272fd7', 'ty